# 🛰️ Module 4: Satellite Image Preprocessing Pipeline

Welcome to **Module 4** of the **Deforestation Detection from Satellite Images** project!

### 🎯 Learning Objectives
1. **Cloud & Shadow Masking**: Detect and filter atmospheric cloud contamination using multispectral thresholds.
2. **Spatial Noise Removal**: Apply edge-preserving median filters to remove speckle noise while maintaining clear logging road boundaries.
3. **Geometric Alignment (Co-Registration)**: Detect and fix sub-pixel and pixel-level temporal misalignment between Before and After images.
4. **Radiometric Normalization**: Standardize surface reflectance into calibrated $[0, 1]$ numerical bounds.
5. **Band Stacking**: Stack multi-temporal spectral channels into unified tensors (e.g. 10-channel tensors) for change detection models.
6. **Image Tiling (Chipping)**: Slice large satellite scenes into uniform model-ready patches with synchronized ground-truth masks.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from scipy.ndimage import shift

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from modules.module_04_preprocessing.preprocessing_pipeline import (
    SatellitePreprocessingPipeline,
    CloudMasker,
    NoiseFilter,
    GeometricAligner,
    RasterNormalizer,
    RasterTiler,
    BandStacker
)

%matplotlib inline
print("✅ Module 4 preprocessing components loaded successfully!")

## 1. Load Raw Multi-Temporal Satellite Pair & Mask

Let's load the Before ($T_1$) and After ($T_2$) 5-band satellite rasters and the corresponding deforestation mask.

In [ ]:
sample_b_path = "../dataset/train/before/scene_train_001.tif"
sample_a_path = "../dataset/train/after/scene_train_001.tif"
sample_m_path = "../dataset/train/masks/scene_train_001.tif"

with rasterio.open(sample_b_path) as src_b:
    raw_before = src_b.read()
with rasterio.open(sample_a_path) as src_a:
    raw_after = src_a.read()
with rasterio.open(sample_m_path) as src_m:
    raw_mask = src_m.read(1)

print(f"Raw Before shape: {raw_before.shape} (Bands, Height, Width)")
print(f"Raw After shape : {raw_after.shape}")
print(f"Raw Mask shape  : {raw_mask.shape}")

## 2. Simulate Real-World Satellite Imperfections

In real-world earth observation, satellite orbits, atmospheric scattering, and cloud cover cause imperfections:
1. **Cloud cover**: Adds intense brightness across visible bands.
2. **Geometric misalignment**: Satellite orbital jitter can shift scenes by several pixels.

In [ ]:
# 1. Add deliberate geometric shift (dy=+3, dx=-4) to After image
sim_after = np.zeros_like(raw_after)
for b in range(5):
    sim_after[b] = shift(raw_after[b], shift=(3.0, -4.0), mode="nearest")

# 2. Add synthetic localized cloud to Before image
yy, xx = np.ogrid[:256, :256]
cloud_patch = ((xx - 70)**2 / (25**2) + (yy - 70)**2 / (20**2) <= 1)
sim_before = raw_before.copy()
sim_before[0:3, cloud_patch] += 0.45

print("Created realistic simulated observation conditions.")

## 3. Execute End-to-End Preprocessing Pipeline

The automated pipeline executes:
`Raw Satellite Image -> Cloud/Noise Removal -> Geometric Alignment -> Band Selection -> Normalization -> Tiling`

In [ ]:
pipeline = SatellitePreprocessingPipeline(
    tile_size=128,
    stride=128,
    normalization_method="robust"
)

results = pipeline.process_pair(sim_before, sim_after, raw_mask)

dy, dx = results["detected_shift_px"]
print(f"✅ Shift detected & corrected : dy = {dy:+.1f} px, dx = {dx:+.1f} px")
print(f"✅ Temporal stack shape       : {results['stacked_temporal'].shape}")
print(f"✅ Total model chips produced : {results['total_chips_extracted']}")

## 4. Visualizing Preprocessing Pipeline Stages

Let's visualize the transformation across each preprocessing stage.

In [ ]:
def make_rgb(bands):
    def strt(b):
        p2, p98 = np.percentile(b, (2, 98))
        return np.clip((b - p2) / (p98 - p2 + 1e-8), 0.0, 1.0)
    return np.stack([strt(bands[2]), strt(bands[1]), strt(bands[0])], axis=-1)

fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)

axes[0, 0].imshow(make_rgb(sim_before))
axes[0, 0].set_title("1. Raw Before (Cloud Contaminated)", fontweight="bold")
axes[0, 0].axis("off")

axes[0, 1].imshow(results["cloud_mask_before"], cmap="gray")
axes[0, 1].set_title("2. Cloud Mask Detected", fontweight="bold", color="darkblue")
axes[0, 1].axis("off")

axes[0, 2].imshow(make_rgb(results["denoised_before"]))
axes[0, 2].set_title("3. Denoised & Cleaned", fontweight="bold", color="darkgreen")
axes[0, 2].axis("off")

axes[1, 0].imshow(make_rgb(sim_after))
axes[1, 0].set_title("4. Misaligned Raw After (Shifted)", fontweight="bold", color="darkred")
axes[1, 0].axis("off")

axes[1, 1].imshow(make_rgb(results["aligned_after"]))
axes[1, 1].set_title(f"5. Co-Registered After (dy={dy:.1f}, dx={dx:.1f})", fontweight="bold", color="green")
axes[1, 1].axis("off")

norm_stack = results["stacked_temporal"]
im_diff = axes[1, 2].imshow(norm_stack[8] - norm_stack[3], cmap="coolwarm", vmin=-0.5, vmax=0.5)
axes[1, 2].set_title("6. Normalized Temporal NIR Difference", fontweight="bold")
axes[1, 2].axis("off")
fig.colorbar(im_diff, ax=axes[1, 2], fraction=0.046, pad=0.04)
plt.show()

## 5. Model-Ready Chipped Tiles (128x128)

The large scene is segmented into uniform training patches with synchronized ground-truth masks.

In [ ]:
chips = results["chips"]
fig, axes = plt.subplots(len(chips), 3, figsize=(10, 3.2 * len(chips)), constrained_layout=True)

for i, chip in enumerate(chips):
    axes[i, 0].imshow(make_rgb(chip["before"]))
    axes[i, 0].set_title(f"Chip {chip['chip_id']}: Before Tile (128x128)")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(make_rgb(chip["after"]))
    axes[i, 1].set_title(f"Chip {chip['chip_id']}: After Tile (128x128)")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(chip["mask"], cmap="viridis", vmin=0, vmax=1)
    axes[i, 2].set_title(f"Chip {chip['chip_id']}: Mask ({chip['deforested_pct']:.1f}% loss)", color="darkred")
    axes[i, 2].axis("off")

plt.show()